# Step 6 — Add the conventional benchmark and expectation revisions
**Runtime → Run all**, then save to GitHub as **05_predictors_benchmark.ipynb**. Standalone: this file visibly includes the tested Step 5 build, so no previous runtime or files are required.

Goal: add the previous-day SF Fed six-month monetary-policy uncertainty baseline and a five-revision measure of Kalshi expected-decision movement. No regressions are run here.

**Definitions fixed before analysis:** the baseline measures broader six-month path uncertainty; Kalshi sigma measures immediate-meeting uncertainty under the authors' construction. Revision RV uses six daily means for the same event, giving five changes: `sqrt(sum(diff(mean_bps)**2))`. A constant adjustment to rate levels cancels in differences; revisions in expected levels equal revisions in expected decisions while the current target stays fixed. These are processed measures, not independently verified fresh quotes.

**Timing:** same previous-day UTC Kalshi convention as Step 5; exact-date baseline joins. A daily-source comparison is not a synchronized 16:00 ET snapshot. No older-date substitution is added. Incomplete revision histories do not remove events from the main sigma models.

**Sources:** [Kalshi replication package](https://github.com/jdkatz21/Prediction_Markets_Public), [SF Fed baseline definitions](https://www.frbsf.org/research-and-insights/data-and-indicators/market-based-monetary-policy-uncertainty/), [Bauer–Lakdawala–Mueller paper](https://www.frbsf.org/economic-research/wp-content/uploads/sites/4/wp2019-12.pdf), [USMPD](https://www.frbsf.org/research-and-insights/data-and-indicators/us-monetary-policy-event-study-database/).

The benchmark source is already a standard deviation, not variance. Its native units are percentage points. Rates convert to basis points by multiplying by 100.

### 1. Reload the three source files

In [1]:
import io
import json
import time
import hashlib
import urllib.request
import urllib.error
from pathlib import Path
import numpy as np
import pandas as pd

# Fixed scope. Refreshing the files can update history; hashes identify each download.
STUDY_START = pd.Timestamp("2022-01-01")
AS_OF_DATE = pd.Timestamp.now(tz="America/New_York").tz_localize(None).normalize()
SOURCES = {
  "moments": "https://kalshi-and-the-rise-of-macro-markets.s3.amazonaws.com/daily_moments_data/daily_moments_fed_levels.csv",
  "distributions": "https://kalshi-and-the-rise-of-macro-markets.s3.amazonaws.com/daily_distribution_data/daily_distributions_fed_levels.csv",
  "usmpd": "https://www.frbsf.org/wp-content/uploads/USMPD.xlsx"
}
source_receipts = []
source_bytes = {}

def download(name, url):
    for attempt in range(3):
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "ResearchProject/1.0"})
            with urllib.request.urlopen(request, timeout=45) as response:
                data = response.read()
                modified = response.headers.get("Last-Modified")
            break
        except (urllib.error.URLError, TimeoutError):
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)
    source_bytes[name] = data
    source_receipts.append({
        "source": name, "url": url, "bytes": len(data),
        "sha256": hashlib.sha256(data).hexdigest(),
        "retrieved_utc": pd.Timestamp.now(tz="UTC").isoformat(),
        "last_modified_http": modified
    })
    return data

moments = pd.read_csv(io.BytesIO(download("moments", SOURCES["moments"])))
distributions = pd.read_csv(io.BytesIO(download("distributions", SOURCES["distributions"])))
workbook = pd.ExcelFile(io.BytesIO(download("usmpd", SOURCES["usmpd"])))
statements = pd.read_excel(workbook, sheet_name="Statements")
print("Downloaded published moments, distributions and USMPD.")
print("No account credentials or API keys are needed.")


Downloaded published moments, distributions and USMPD.
No account credentials or API keys are needed.


### 2. Validate schemas and scheduled event keys

In [2]:
required_m = {"date", "contract_preamble", "mean", "variance", "daily_volume"}
required_d = {"date", "contract_preamble", "strike", "probability"}
assert required_m <= set(moments.columns), "Moments schema changed"
assert required_d <= set(distributions.columns), "Distribution schema changed"
assert {"Date", "Unscheduled", "UST2Y"} <= set(statements.columns), "USMPD schema changed"

for frame in [moments, distributions]:
    frame["date"] = pd.to_datetime(frame["date"], errors="raise")
    assert frame["date"].notna().all()
    frame["event_key"] = frame["contract_preamble"].str.replace(r"^KX", "", regex=True)
assert not moments.duplicated(["event_key", "date"]).any(), "Duplicate event/date moments"
assert not distributions.duplicated(["event_key", "date", "strike"]).any(), "Duplicate bins"
for column in ["mean", "variance", "daily_volume"]:
    moments[column] = pd.to_numeric(moments[column], errors="raise")
for column in ["strike", "probability"]:
    distributions[column] = pd.to_numeric(distributions[column], errors="raise")
assert moments["variance"].dropna().ge(-1e-12).all(), "Negative variance"
assert distributions["probability"].dropna().between(0, 100 + 1e-8).all()

statements["Date"] = pd.to_datetime(statements["Date"], errors="raise")
assert statements["Date"].is_unique
assert statements["Unscheduled"].isin([0, 1]).all()
statements["UST2Y"] = pd.to_numeric(statements["UST2Y"], errors="raise")
# Never use the source's expiry_date as an official meeting date:
# the authors calculate it from the last observed trade day.
events = statements.loc[
    statements["Unscheduled"].eq(0) &
    statements["Date"].ge(STUDY_START) &
    statements["Date"].le(AS_OF_DATE), ["Date", "UST2Y"]
].rename(columns={"Date": "meeting_date", "UST2Y": "ust2y_change_pp"}).copy()
events["event_key"] = events["meeting_date"].map(
    lambda d: "FED-" + d.strftime("%y%b").upper()
)
assert events["event_key"].is_unique, "Two meetings share an event month; inspect mapping"
events["observation_date"] = events["meeting_date"] - pd.offsets.BDay(1)
assert events["observation_date"].lt(events["meeting_date"]).all()
print("Kalshi observation range:", moments["date"].min().date(), "to", moments["date"].max().date())
print("Latest USMPD statement:", statements["Date"].max().date())
print("Scheduled meetings in scope:", len(events))
print("Event mapping uses official USMPD meeting dates and month identifiers.")
print("Previous weekday convention: selected FOMC cutoffs must also be checked for holidays.")


Kalshi observation range: 2021-07-02 to 2026-10-16
Latest USMPD statement: 2026-09-16
Scheduled meetings in scope: 38
Event mapping uses official USMPD meeting dates and month identifiers.
Previous weekday convention: selected FOMC cutoffs must also be checked for holidays.


### 3. Reconcile the source distributions and audit joins

In [3]:
# Join exactly one published daily observation to each scheduled event.
audit = events.merge(
    moments[["event_key", "date", "mean", "variance", "daily_volume"]],
    left_on=["event_key", "observation_date"],
    right_on=["event_key", "date"], how="left", validate="one_to_one",
    indicator="moments_match"
)
assert len(audit) == len(events)
targets = events[["event_key", "observation_date"]]
bins = distributions.merge(
    targets, left_on=["event_key", "date"],
    right_on=["event_key", "observation_date"],
    how="inner", validate="many_to_one"
)
checks = []
for (event_key, date), group in bins.groupby(["event_key", "date"]):
    p = group["probability"].to_numpy(dtype=float) / 100
    x = group["strike"].to_numpy(dtype=float)
    finite = np.isfinite(p).all() and np.isfinite(x).all()
    total = p.sum()
    if finite and total > 0:
        weights = p / total
        mean_unadjusted = np.sum(weights * x)
        var_rebuilt = np.sum(weights * (x - mean_unadjusted) ** 2)
    else:
        mean_unadjusted = var_rebuilt = np.nan
    checks.append({
        "event_key": event_key, "observation_date": date,
        "probability_sum": total, "bins": len(group),
        "rebuilt_variance_pp2": var_rebuilt,
        "rebuilt_mean_pp": mean_unadjusted + 0.125,
        "distribution_valid": bool(finite and np.isclose(total, 1, atol=1e-8))
    })
checks = pd.DataFrame(checks, columns=[
    "event_key", "observation_date", "probability_sum", "bins",
    "rebuilt_variance_pp2", "rebuilt_mean_pp", "distribution_valid"
])
audit = audit.merge(checks, on=["event_key", "observation_date"],
                    how="left", validate="one_to_one")
audit["variance_matches"] = np.isclose(
    audit["variance"], audit["rebuilt_variance_pp2"], atol=1e-10, rtol=1e-7
)
audit["mean_matches"] = np.isclose(
    audit["mean"], audit["rebuilt_mean_pp"], atol=1e-8, rtol=1e-7
)
audit["ust2y_signed_bps"] = audit["ust2y_change_pp"] * 100
audit["abs_ust2y_bps"] = audit["ust2y_signed_bps"].abs()
audit["kalshi_sigma_bps"] = np.sqrt(audit["variance"].clip(lower=0)) * 100

def inclusion_reason(row):
    if row["moments_match"] != "both":
        return "exclude: missing exact previous-day moments"
    if not np.isfinite(row["ust2y_signed_bps"]):
        return "exclude: missing Treasury outcome"
    if not np.isfinite(row["variance"]) or row["variance"] < -1e-12:
        return "exclude: invalid variance"
    if not (pd.notna(row["distribution_valid"]) and row["distribution_valid"]):
        return "exclude: missing or invalid distribution"
    if not row["variance_matches"] or not row["mean_matches"]:
        return "exclude: published moments do not reconcile"
    return "include: published-method candidate"

audit["inclusion_reason"] = audit.apply(inclusion_reason, axis=1)
audit["included"] = audit["inclusion_reason"].eq("include: published-method candidate")
print(audit[["meeting_date", "observation_date", "kalshi_sigma_bps",
             "abs_ust2y_bps", "inclusion_reason"]].round(3).to_string(index=False))
print("\nYear-level coverage:")
print(audit.groupby(audit["meeting_date"].dt.year).agg(
    scheduled=("included", "size"), included=("included", "sum")
).to_string())


meeting_date observation_date  kalshi_sigma_bps  abs_ust2y_bps                            inclusion_reason
  2022-01-26       2022-01-25             4.307          0.190         include: published-method candidate
  2022-03-16       2022-03-15             7.468         10.900         include: published-method candidate
  2022-05-04       2022-05-03             7.970          4.700         include: published-method candidate
  2022-06-15       2022-06-14            13.216          5.050         include: published-method candidate
  2022-07-27       2022-07-26             8.183          0.200         include: published-method candidate
  2022-09-21       2022-09-20             6.559          9.390         include: published-method candidate
  2022-11-02       2022-11-01             5.595          8.720         include: published-method candidate
  2022-12-14       2022-12-13             8.480          8.580         include: published-method candidate
  2023-02-01       2023-01-31        

### 4. Rebuild the 37-event candidate dataset

In [4]:
dataset = audit.loc[audit["included"], [
    "meeting_date", "event_key", "observation_date", "mean",
    "kalshi_sigma_bps", "ust2y_signed_bps", "abs_ust2y_bps",
    "daily_volume", "probability_sum", "bins"
]].rename(columns={
    "mean": "authors_adjusted_mean_rate_pct",
    "daily_volume": "authors_daily_volume"
}).sort_values("meeting_date").reset_index(drop=True)

assert dataset["meeting_date"].is_unique
assert dataset["observation_date"].lt(dataset["meeting_date"]).all()
assert np.isfinite(dataset[["kalshi_sigma_bps", "abs_ust2y_bps"]]).all().all()
assert dataset["kalshi_sigma_bps"].ge(0).all()
print("Reconciled candidate events:", len(dataset))
print("Excluded scheduled events:", int((~audit["included"]).sum()))
print("Zero constructed uncertainty:", int(dataset["kalshi_sigma_bps"].eq(0).sum()))
print("\nDescriptive ranges (basis points):")
print(dataset[["kalshi_sigma_bps", "abs_ust2y_bps"]].describe().round(3).to_string())
print("\nLIMITS:")
print("- This is a dataset using the authors' processed measures, not an exact raw distribution.")
print("- Daily prices can be carried forward; daily volume does not establish age of every strike.")
print("- UTC calendar-day aggregation replaces the original 16:00 Eastern cutoff.")
print("- Reconciled means include the authors' 0.125 percentage-point adjustment.")
print("- Constructed zero variance does not establish zero true uncertainty.")
print("- Corrections, tail conventions and retrospective processing require sensitivity checks.")
print("- No claims of predictive skill or causality are made; no regressions have been run.")
print("- Policy surprise, revision volatility and conventional benchmark are not yet built.")


Reconciled candidate events: 37
Excluded scheduled events: 1
Zero constructed uncertainty: 5

Descriptive ranges (basis points):
       kalshi_sigma_bps  abs_ust2y_bps
count            37.000         37.000
mean              5.088          4.010
std               3.533          3.468
min               0.000          0.000
25%               2.512          1.040
50%               4.307          2.770
75%               7.892          6.525
max              13.216         11.775

LIMITS:
- This is a dataset using the authors' processed measures, not an exact raw distribution.
- Daily prices can be carried forward; daily volume does not establish age of every strike.
- UTC calendar-day aggregation replaces the original 16:00 Eastern cutoff.
- Reconciled means include the authors' 0.125 percentage-point adjustment.
- Constructed zero variance does not establish zero true uncertainty.
- Corrections, tail conventions and retrospective processing require sensitivity checks.
- No claims of predi

### 5. Add the six-month uncertainty baseline

In [5]:
MPU_URL = "https://www.frbsf.org/wp-content/uploads/market-based-monetary-policy-uncertainty-data.xlsx"
mpu_bytes = download("mpu", MPU_URL)
mpu_book = pd.ExcelFile(io.BytesIO(mpu_bytes))
assert {"readme", "mpu"} <= set(mpu_book.sheet_names)
mpu = pd.read_excel(mpu_book, sheet_name="mpu")
assert {"Date", "6 months"} <= set(mpu.columns), "Benchmark schema changed"
mpu["Date"] = pd.to_datetime(mpu["Date"], errors="raise")
mpu["6 months"] = pd.to_numeric(mpu["6 months"], errors="raise")
assert mpu["Date"].is_unique and mpu["Date"].notna().all()
assert mpu["6 months"].dropna().ge(0).all()

# The source is already a standard deviation, in percentage points.
# Multiply by 100 for bps; do NOT take another square root.
mpu["mpu6m_bps"] = mpu["6 months"] * 100
enriched = dataset.merge(
    mpu[["Date", "mpu6m_bps"]].rename(columns={"Date": "mpu_observation_date"}),
    left_on="observation_date", right_on="mpu_observation_date",
    how="left", validate="one_to_one"
)
assert len(enriched) == len(dataset)
assert enriched["mpu_observation_date"].dropna().lt(
    enriched.loc[enriched["mpu_observation_date"].notna(), "meeting_date"]
).all()
print("Latest benchmark observation:", mpu["Date"].max().date())
print("Exact previous-day benchmark matches:", enriched["mpu6m_bps"].notna().sum(),
      "of", len(enriched))
print("No backward-as-of substitution or interpolation applied.")
print("Benchmark covers six-month policy-path uncertainty, not immediate-decision uncertainty.")
print("Underlying derivatives switch from Eurodollar to SOFR in October 2022.")


Latest benchmark observation: 2026-10-09
Exact previous-day benchmark matches: 37 of 37
No backward-as-of substitution or interpolation applied.
Benchmark covers six-month policy-path uncertainty, not immediate-decision uncertainty.
Underlying derivatives switch from Eurodollar to SOFR in October 2022.


### 6. Calculate five revisions from six same-meeting daily levels
All observation dates and source volumes are preserved in an audit. We check the weekday windows against federal holidays and Good Friday; any flagged future window is excluded from RV pending calendar review. No missing daily means are replaced with zero.

In [6]:
from pandas.tseries.holiday import USFederalHolidayCalendar
from dateutil.easter import easter

# Five daily revisions need SIX level observations for the SAME meeting.
# In this sample, check that the selected weekdays contain no federal
# holidays or Good Friday. A future flagged window needs calendar review.
years = range(int(enriched["meeting_date"].dt.year.min()) - 1,
              int(enriched["meeting_date"].dt.year.max()) + 1)
closed_dates = USFederalHolidayCalendar().holidays(
    start=f"{min(years)}-01-01", end=f"{max(years)}-12-31"
).union(pd.DatetimeIndex([
    pd.Timestamp(easter(year)) - pd.Timedelta(days=2) for year in years
]))
revision_rows = []
window_rows = []
indexed = moments.set_index(["event_key", "date"]).sort_index()
for _, event in enriched.iterrows():
    dates = pd.bdate_range(end=event["observation_date"], periods=6)
    holiday_flag = bool(dates.isin(closed_dates).any())
    key = pd.MultiIndex.from_product(
        [[event["event_key"]], dates], names=["event_key", "date"]
    )
    window = indexed.reindex(key)
    mean_levels = window["mean"].to_numpy(dtype=float) * 100
    complete = bool(np.isfinite(mean_levels).all()) and not holiday_flag
    changes = np.diff(mean_levels) if complete else np.full(5, np.nan)
    rv5 = float(np.sqrt(np.sum(changes ** 2))) if complete else np.nan
    mean_abs = float(np.mean(np.abs(changes))) if complete else np.nan
    zero_volume_days = int(window["daily_volume"].eq(0).sum())
    revision_rows.append({
        "meeting_date": event["meeting_date"],
        "rv5_bps": rv5,
        "mean_abs_daily_revision_bps": mean_abs,
        "revision_level_observations": int(np.isfinite(mean_levels).sum()),
        "revision_window_complete": complete,
        "revision_calendar_review": holiday_flag,
        "window_zero_volume_days": zero_volume_days
    })
    for position, date in enumerate(dates):
        window_rows.append({
            "meeting_date": event["meeting_date"], "event_key": event["event_key"],
            "observation_date": date, "authors_adjusted_mean_bps": mean_levels[position],
            "change_from_previous_bps": changes[position-1] if position else np.nan,
            "authors_daily_volume": window["daily_volume"].iloc[position],
            "window_complete": complete
        })
revision_audit = pd.DataFrame(revision_rows)
revision_windows = pd.DataFrame(window_rows)
assert revision_windows["observation_date"].lt(revision_windows["meeting_date"]).all()
assert not revision_windows.duplicated(
    ["meeting_date", "observation_date"]
).any()
enriched = enriched.merge(revision_audit, on="meeting_date",
                          how="left", validate="one_to_one")
assert len(enriched) == len(dataset)
print("Complete five-revision windows:",
      int(enriched["revision_window_complete"].sum()), "of", len(enriched))
print("Calendar-review windows:", int(enriched["revision_calendar_review"].sum()))
print("Windows with zero published daily volume:",
      int(enriched["window_zero_volume_days"].gt(0).sum()))
print("Missing levels remain missing; no partial-window RV or zero filling.")
print("Published forward-filled levels may still conceal stale underlying strikes.")


Complete five-revision windows: 37 of 37
Calendar-review windows: 0
Windows with zero published daily volume: 6
Missing levels remain missing; no partial-window RV or zero filling.
Published forward-filled levels may still conceal stale underlying strikes.


### 7. Freeze model-specific sample flags and preserve outputs
Nested baseline comparisons must use identical observations. Revision models are separate from sigma models. This notebook prepares variables and audit evidence; it does not establish forecasting ability or causal effects.

In [7]:
# Fix the specifications before inspecting regressions.
# A: absolute yield move ~ decision sigma
# B0/B1: baseline alone versus baseline + decision sigma, on IDENTICAL rows
# P0/P1: baseline alone versus baseline + revision RV, estimated in parallel
# Do not put sigma, RV and many additional controls into a kitchen-sink model.
enriched["sample_A"] = np.isfinite(
    enriched[["abs_ust2y_bps", "kalshi_sigma_bps"]]
).all(axis=1)
enriched["sample_B"] = enriched["sample_A"] & np.isfinite(enriched["mpu6m_bps"])
enriched["sample_revision"] = (
    np.isfinite(enriched[["abs_ust2y_bps", "mpu6m_bps", "rv5_bps"]]).all(axis=1)
    & enriched["revision_window_complete"]
)
print(enriched[[
    "meeting_date", "kalshi_sigma_bps", "mpu6m_bps", "rv5_bps",
    "revision_level_observations", "sample_B", "sample_revision"
]].round(3).to_string(index=False))
print("\nFrozen candidate sample sizes:")
print("A, sigma only:", int(enriched["sample_A"].sum()))
print("B, baseline versus baseline + sigma:", int(enriched["sample_B"].sum()))
print("Parallel baseline versus baseline + RV:", int(enriched["sample_revision"].sum()))
print("\nPre-specified sensitivity checks:")
print("- Spearman correlation, HC3 inference and leave-one-event-out sensitivity")
print("- Zero-constructed-sigma exclusions as sensitivity only, not significance-based selection")
print("- Alternate statement/full-event windows and reasonable measurement conventions")
print("- Eurodollar/SOFR source transition is documented; avoid overfitting regime splits")
print("No regressions or significance searches have been run.")
print("Policy-surprise analysis remains separate; no realized surprise is an ex-ante predictor.")

OUTPUT = Path("research_outputs")
OUTPUT.mkdir(exist_ok=True)
enriched.to_csv(OUTPUT / "meeting_dataset_with_predictors.csv", index=False)
revision_windows.to_csv(OUTPUT / "revision_window_audit.csv", index=False)
revision_audit.to_csv(OUTPUT / "revision_completeness_audit.csv", index=False)
metadata = {
    "sources": source_receipts,
    "baseline": "Previous-day SF Fed 6-month market-based uncertainty; native pp * 100 = bps",
    "revision": "sqrt(sum of five squared revisions across six pre-announcement daily means)); means * 100 = bps",
    "timing": "Previous UTC day; weekday windows checked against federal holidays and Good Friday",
    "missingness": "Exact-date joins only; no additional forward filling; incomplete revision windows excluded only from revision models",
    "method_limits": [
        "Authors' daily Kalshi levels already carry forward stale prices and apply distribution repairs",
        "Daily aggregate volume does not identify exact age of every underlying strike",
        "Historical files are current-vintage processed data, not a verified real-time vintage archive",
        "Different market horizons and daily closing times limit direct comparability"
    ],
    "candidate_model_samples": {
        "A": int(enriched["sample_A"].sum()), "B": int(enriched["sample_B"].sum()),
        "revision_parallel": int(enriched["sample_revision"].sum())
    }
}
(OUTPUT / "predictor_provenance.json").write_text(json.dumps(metadata, indent=2))
print("\nSaved derived dataset, two revision audits and provenance under research_outputs/.")
print("Raw source files are not exported.")
print("Save to ResearchProject / main as 05_predictors_benchmark.ipynb.")


meeting_date  kalshi_sigma_bps  mpu6m_bps  rv5_bps  revision_level_observations  sample_B  sample_revision
  2022-01-26             4.307     41.713    1.052                            6      True             True
  2022-03-16             7.468     83.628    1.593                            6      True             True
  2022-05-04             7.970    102.297    3.287                            6      True             True
  2022-06-15            13.216    123.957   15.622                            6      True             True
  2022-07-27             8.183    108.859    0.920                            6      True             True
  2022-09-21             6.559     92.942    2.134                            6      True             True
  2022-11-02             5.595     90.428    1.112                            6      True             True
  2022-12-14             8.480     72.310    3.894                            6      True             True
  2023-02-01             3.571     53